# nqueens-recog

Detect and parse a colour grid from a Queens puzzle screenshot.
Recognition is purely colour-based (no OCR): k-means quantisation assigns each
cell to one of the *n* region colours, then the nearest game-palette entry is
looked up by squared RGB distance.

In [2]:
import sys
import os

# Make the src package importable when running from the notebooks/ directory
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'src'))

In [ ]:
from pathlib import Path
from nqueens_recog.grid_reader import read_grid
from nqueens_recog.palette import grid_to_letters

IMG = Path("../img/puzzle-687.png")
grid = read_grid(str(IMG))
print(f"Detected grid: {grid.rows} rows × {grid.cols} cols, {grid.rows} colours")

Hello, World! Welcome to nqueens-recog.


In [ ]:
for row in grid_to_letters(grid):
    print(" ".join(row))

Hello, N-Queens Solver! Welcome to nqueens-recog.


## Visualisation

Render each cell with its detected colour. The letter from the nearest palette
match is overlaid on each tile.

In [ ]:
import matplotlib.pyplot as plt
from nqueens_recog.grid_reader import Grid
from nqueens_recog.palette import nearest_letter

In [ ]:
def show_grid(grid: Grid, title: str = "Recognised Grid", figsize: tuple = (14, 14)) -> None:
    """Render the grid with each tile's detected colour and palette letter."""
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(0, grid.cols)
    ax.set_ylim(0, grid.rows)
    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title, fontsize=14, pad=10)

    fs = max(5, int(180 / max(grid.rows, grid.cols)))

    for row_tiles in grid.tiles:
        for t in row_tiles:
            r, g, b = t.color_rgb
            face = (r / 255, g / 255, b / 255)
            ax.add_patch(plt.Rectangle(
                (t.col, t.row), 1, 1,
                facecolor=face, edgecolor="#222222", linewidth=0.4,
            ))
            letter = nearest_letter(t.color_rgb) or "?"
            lum = 0.299 * r + 0.587 * g + 0.114 * b
            ax.text(
                t.col + 0.5, t.row + 0.5, letter,
                ha="center", va="center", fontsize=fs, fontweight="bold",
                color="black" if lum > 100 else "white",
            )

    plt.tight_layout()
    plt.show()


show_grid(grid)

In [ ]:
# ── Inspect the raw data structure ─────────────────────────────────────────
print("Tile at (0, 0):", grid[0, 0])
print("Tile at (0, 1):", grid[0, 1])
print()
print("Color grid (first 3 rows, first 4 cols):")
for row in grid.color_grid()[:3]:
    print(" ", row[:4])